# USP5 (53 senyawa): evaluasi dua tingkat + baseline
- Tingkat 1 (interpolasi): repeated random 5-fold CV
- Tingkat 2 (ekstrapolasi): leave-cluster-out (Butina clustering pada ECFP4)
- Model: baseline mean, RF-ECFP, SVR-Tanimoto, (opsional) GraphConvModel ensemble seed
Kolom CSV yang dipakai: `Smiles` dan `pKd`.

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ---- PENGATURAN ----
CSV_PATH = "USP5_MANN_53.csv"   # di Colab: upload dulu, atau pakai URL raw GitHub
SMILES_COL, Y_COL = "Smiles", "pKd"
BUTINA_CUTOFF = 0.6             # jarak = 1 - Tanimoto. Kecilkan -> cluster lebih banyak & lebih kecil
N_FOLDS, N_REPEATS = 5, 10
USE_GNN = False                 # True jika DeepChem sudah terpasang
GNN_SEEDS, GNN_REPEATS = 3, 3   # GNN lambat: ensemble 3 seed, 3 pengulangan
GNN_EPOCHS, GNN_BATCH = 40, 16  # samakan batch dengan notebook GraphConv-mu
SEED0 = 42

In [ ]:
df = pd.read_csv(CSV_PATH).dropna(subset=[SMILES_COL, Y_COL]).reset_index(drop=True)
y = df[Y_COL].values.astype(float)
n = len(df)
print(f"n = {n}")
print(f"pKd: min {y.min():.2f} | max {y.max():.2f} | mean {y.mean():.2f} | SD {y.std(ddof=1):.2f}")

In [ ]:
# ---- Fingerprint ECFP4 + Butina clustering (butuh RDKit) ----
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from rdkit.Chem.MolStandardize import rdMolStandardize

# Standardisasi: ambil fragmen terbesar (buang K+, Na+, dst) dan netralkan muatan,
# lalu gabungkan duplikat parent (pKd dirata-rata) supaya "kembar" tidak bocor antar fold.
_lfc, _unch = rdMolStandardize.LargestFragmentChooser(), rdMolStandardize.Uncharger()
def parent_smiles(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(_unch.uncharge(_lfc.choose(m)))

n_before = len(df)
df[SMILES_COL] = df[SMILES_COL].apply(parent_smiles)
df = df.groupby(SMILES_COL, as_index=False)[Y_COL].mean()
y, n = df[Y_COL].values.astype(float), len(df)
print(f"Setelah standardisasi & deduplikasi parent: {n_before} -> {n} senyawa")

mols = [Chem.MolFromSmiles(s) for s in df[SMILES_COL]]
assert all(m is not None for m in mols), "Ada SMILES yang gagal dibaca"
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(m) for m in mols]
X = np.array([list(fp) for fp in fps], dtype=np.uint8)

def butina_groups(fps, cutoff):
    dists = []
    for i in range(1, len(fps)):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1 - s for s in sims])
    clusters = Butina.ClusterData(dists, len(fps), cutoff, isDistData=True)
    groups = np.zeros(len(fps), dtype=int)
    for cid, members in enumerate(clusters):
        groups[list(members)] = cid
    return groups

groups = butina_groups(fps, BUTINA_CUTOFF)
sizes = pd.Series(groups).value_counts().sort_values(ascending=False)
print(f"Jumlah cluster: {len(sizes)} | ukuran: {sizes.values.tolist()}")
print(f"Cluster terbesar = {sizes.iloc[0] / n:.0%} dari data")

# Activity cliff: pasangan sangat mirip (Tanimoto > 0.8) tetapi selisih pKd > 1
S = np.array([DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps])
iu = np.triu_indices(n, k=1)
cliff = (S[iu] > 0.8) & (np.abs(y[iu[0]] - y[iu[1]]) > 1.0)
print(f"Pasangan activity cliff (sim>0.8, |dpKd|>1): {cliff.sum()}")

In [ ]:
# ---- Model ----
def tanimoto_kernel(A, B):
    inter = A.astype(float) @ B.astype(float).T
    union = A.sum(1)[:, None] + B.sum(1)[None, :] - inter
    return inter / np.maximum(union, 1e-9)

def m_mean(tr, te, seed):
    return np.full(len(te), y[tr].mean())

def m_rf(tr, te, seed):
    m = RandomForestRegressor(n_estimators=500, random_state=seed, n_jobs=-1)
    return m.fit(X[tr], y[tr]).predict(X[te])

def m_svr(tr, te, seed):
    m = SVR(kernel="precomputed", C=5.0, epsilon=0.1)
    m.fit(tanimoto_kernel(X[tr], X[tr]), y[tr])
    return m.predict(tanimoto_kernel(X[te], X[tr]))

def m_gnn(tr, te, seed):
    import deepchem as dc, tensorflow as tf
    feat = dc.feat.ConvMolFeaturizer()
    smi = df[SMILES_COL].values
    ds_tr = dc.data.NumpyDataset(feat.featurize(smi[tr]), y[tr].reshape(-1, 1))
    ds_te = dc.data.NumpyDataset(feat.featurize(smi[te]))
    preds = []
    for k in range(GNN_SEEDS):
        tf.random.set_seed(seed * 100 + k)
        model = dc.models.GraphConvModel(
            n_tasks=1, graph_conv_layers=[64, 64], dense_layer_size=128,
            dropout=0.0, mode="regression", learning_rate=0.001, batch_size=GNN_BATCH)
        model.fit(ds_tr, nb_epoch=GNN_EPOCHS)
        preds.append(model.predict(ds_te).reshape(-1))
    return np.mean(preds, axis=0)

MODELS = {"Mean (baseline)": (m_mean, N_REPEATS), "RF-ECFP": (m_rf, N_REPEATS),
          "SVR-Tanimoto": (m_svr, N_REPEATS)}
if USE_GNN:
    MODELS["GNN (ensemble)"] = (m_gnn, GNN_REPEATS)

In [ ]:
# ---- Pembagi data ----
def random_folds(n, k, seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    return [np.sort(f) for f in np.array_split(perm, k)]

def cluster_folds(groups, k, seed):
    """Satu cluster utuh selalu berada di satu fold (tidak bocor ke training)."""
    rng = np.random.RandomState(seed)
    ids = rng.permutation(np.unique(groups))
    ids = sorted(ids, key=lambda g: -(groups == g).sum())  # besar dulu, urutan sama-ukuran acak
    folds = [[] for _ in range(k)]
    for g in ids:
        smallest = min(range(k), key=lambda i: len(folds[i]))
        folds[smallest].extend(np.where(groups == g)[0].tolist())
    return [np.array(sorted(f)) for f in folds if len(f) > 0]

def metrics(y_true, y_pred):
    rho = spearmanr(y_true, y_pred).correlation if np.std(y_pred) > 0 else np.nan
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"R2": r2_score(y_true, y_pred), "RMSE": rmse,
            "MAE": mean_absolute_error(y_true, y_pred), "Spearman": rho}

def evaluate(split_fn, label):
    rows, oof_store = [], {}
    for name, (fn, reps) in MODELS.items():
        per_rep, last_oof = [], None
        for r in range(reps):
            oof = np.full(n, np.nan)
            for te in split_fn(SEED0 + r):
                tr = np.setdiff1d(np.arange(n), te)
                oof[te] = fn(tr, te, SEED0 + r)
            per_rep.append(metrics(y, oof))
            last_oof = oof
        d = pd.DataFrame(per_rep)
        rows.append({"Tingkat": label, "Model": name,
                     **{f"{c} (mean±SD)": f"{d[c].mean():.3f} ± {d[c].std(ddof=1):.3f}" for c in d.columns}})
        oof_store[name] = last_oof
    return pd.DataFrame(rows), oof_store

In [ ]:
res_a, oof_a = evaluate(lambda s: random_folds(n, N_FOLDS, s), "1. Interpolasi (random CV)")
res_b, oof_b = evaluate(lambda s: cluster_folds(groups, N_FOLDS, s), "2. Ekstrapolasi (leave-cluster-out)")
summary = pd.concat([res_a, res_b], ignore_index=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
print(summary.to_string(index=False))
print(f"\nSD pKd data = {y.std(ddof=1):.3f} -> bandingkan dengan RMSE (RMSE >= SD berarti model tidak lebih baik dari menebak rata-rata)")
summary.to_csv("hasil_evaluasi_USP5_53.csv", index=False)

In [ ]:
# ---- Plot prediksi vs eksperimen (pengulangan terakhir) ----
import matplotlib.pyplot as plt
best = [m for m in MODELS if m != "Mean (baseline)"]
fig, axes = plt.subplots(len(best), 2, figsize=(9, 4 * len(best)), squeeze=False)
lo, hi = y.min() - 0.3, y.max() + 0.3
for i, name in enumerate(best):
    for j, (oof, title) in enumerate([(oof_a[name], "Random CV"), (oof_b[name], "Leave-cluster-out")]):
        ax = axes[i, j]
        ax.scatter(y, oof, c=groups, cmap="tab20", s=25)
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_title(f"{name} | {title} | R2={r2_score(y, oof):.2f}")
        ax.set_xlabel("pKd eksperimen"); ax.set_ylabel("pKd prediksi")
plt.tight_layout(); plt.savefig("scatter_USP5_53.png", dpi=150); plt.show()